In [23]:
import pandas as pd
import json
import ollama
from collections import defaultdict

# ====================
# Load Job Descriptions
# ====================
jd_df = pd.read_excel("jobdescription.xlsx")
jd_df["job_id"] = jd_df.index.astype(int)

# ====================
# Load Resumes
# ====================
resume = pd.read_csv("resumes.csv")
resume_df = resume[[
    "Uniq Id", "Resume Title", "Introduction", 
    "Work Experience", "Education", "Skills", 
    "Additional Information"
]]

len(jd_df), len(resume_df)


(1231, 10)

In [24]:
def build_resume_profile(row):
    return f"""
Resume Title: {row['Resume Title']}

Introduction:
{row['Introduction']}

Work Experience:
{row['Work Experience']}

Education:
{row['Education']}

Skills:
{row['Skills']}

Additional Information:
{row['Additional Information']}
"""


In [25]:
def build_prompt(resume_text, jd_batch):
    jd_section = ""
    for idx, row in jd_batch.iterrows():
        jd_section += f"""
---
Job ID: {row['job_id']}
Job Title: {row['job_title']}
Location: {row['location_cleaned']}
Description:
{row['job_description']}
"""

    # IMPORTANT: force deepseek to output ONLY JSON
    prompt = f"""
You are a senior recruiter.

Analyze resume & job descriptions and score each job from 0 to 1.

Focus weights:
- Skills match (highest)
- Experience relevance
- Education fit
- Location match (lowest)

Return STRICT JSON. NO explanation outside JSON. NO chain-of-thought.

JSON format:

[
  {{
    "job_id": 123,
    "score": 0.0
  }},
  ...
]

Resume:
{resume_text}

Job Descriptions:
{jd_section}

Return ONLY Top 5 results as JSON.
"""

    return prompt


In [26]:
def safe_parse_items(raw_list):
    fixed = []
    for item in raw_list:
        # Case 1: item is string
        if isinstance(item, str):
            # try extract integer job_id from the string
            import re
            ids = re.findall(r'\d+', item)
            score = re.findall(r'\d+\.\d+', item)
            if ids:
                fixed.append({
                    "job_id": int(ids[0]),
                    "score": float(score[0]) if score else 0.0
                })
        # Case 2: item is dict but missing job_id
        elif isinstance(item, dict):
            jid = (
                item.get("job_id") or
                item.get("id") or
                item.get("job") or
                item.get("Job ID") or
                item.get("JobID")
            )
            score = item.get("score", 0.0)
            if jid is not None:
                fixed.append({
                    "job_id": int(jid),
                    "score": float(score)
                })
    return fixed


In [27]:
def rank_jd_batch(resume_text, jd_batch):
    prompt = build_prompt(resume_text, jd_batch)

    resp = ollama.chat(
        model="deepseek-r1:8b",
        messages=[{"role": "user", "content": prompt}]
    )

    raw = resp["message"]["content"]

    # Remove unwanted deepseek <think> sections
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()

    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        parsed = json.loads(raw)
        parsed = safe_parse_items(parsed)
        return parsed
    except:
        print("RAW ERROR:", raw)
        return []



In [28]:
def chunk_df(df, batch_size=5):
    for i in range(0, len(df), batch_size):
        yield df.iloc[i:i + batch_size]


In [29]:
def match_resume_to_jobs(resume_text, jd_df, batch_size=5):
    score_map = defaultdict(float)

    for jd_batch in chunk_df(jd_df, batch_size):
        results = rank_jd_batch(resume_text, jd_batch)
        for item in results:
            job_id = int(item["job_id"])
            score = float(item["score"])
            score_map[job_id] = max(score_map[job_id], score)

    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)
    return ranked[:3]


In [30]:
output_rows = []

for idx, row in resume_df.iterrows():
    resume_id = row["Uniq Id"]
    resume_text = build_resume_profile(row)

    top3 = match_resume_to_jobs(resume_text, jd_df)

    rec = {"resume_id": resume_id}

    for i, (job_id, score) in enumerate(top3, start=1):
        job_row = jd_df.loc[jd_df["job_id"] == job_id].iloc[0]

        rec[f"top{i}_job_id"] = job_id
        rec[f"top{i}_job_title"] = job_row["job_title"]
        rec[f"top{i}_job_location"] = job_row["location_cleaned"]
        rec[f"top{i}_score"] = score

    output_rows.append(rec)


RAW ERROR: Based on the provided job descriptions and the inferred skills from the resume (such as CRM, contract negotiation, and technical aptitude), the best match is Job ID 13: Content Developer. This position aligns well with the resume's strengths in communication, technical skills, and customer relationship management, even though there is a mismatch in the education requirement (resume has a B.A. in History, while the job requires an engineering background). The scores for the other jobs are lower, indicating less suitability.

**Job Recommendation:** Apply for Job ID 13 (Content Developer).

**Summary of Matches (for reference):**
- Job ID 10 (Wealth Plan Specialist): Score 0.20 (lowest match due to finance and education mismatch)
- Job ID 11 (Reference Solutions Architect): Score 0.27 (good for technical skills but poor education fit)
- Job ID 12 (Reference Solutions Architect): Score 0.27 (similar to ID 11, but AWS scripting may not align perfectly)
- Job ID 13 (Content Devel

KeyboardInterrupt: 

In [ ]:
result_df = pd.DataFrame(output_rows)
# result_df.to_excel("resume_job_matching_results_local_llm.xlsx", index=False)

result_df.head()
